In [6]:
from vllm import LLM

# Initialize the vLLM engine.
llm = LLM(model="../weights/Qwen2.5-VL-3B-Instruct-AWQ", quantization="AWQ")

INFO 10-22 10:34:09 [utils.py:233] non-default args: {'disable_log_stats': True, 'quantization': 'AWQ', 'model': '../weights/Qwen2.5-VL-3B-Instruct-AWQ'}
INFO 10-22 10:34:09 [model.py:547] Resolved architecture: Qwen2_5_VLForConditionalGeneration
INFO 10-22 10:34:09 [model.py:1510] Using max model len 128000
INFO 10-22 10:34:09 [awq_marlin.py:123] Detected that the model can run with awq_marlin, however you specified quantization=awq explicitly, so forcing awq. Use quantization=awq_marlin for faster inference
INFO 10-22 10:34:09 [scheduler.py:205] Chunked prefill is enabled with max_num_batched_tokens=8192.


ValidationError: 1 validation error for VllmConfig
  Value error, torch.bfloat16 is not supported for quantization method awq. Supported dtypes: [torch.float16] [type=value_error, input_value=ArgsKwargs((), {'model_co...additional_config': {}}), input_type=ArgsKwargs]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

In [1]:
# -*- coding: utf-8 -*-
import torch
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor
from vllm import LLM, SamplingParams

import os
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'

def prepare_inputs_for_vllm(messages, processor):
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    # qwen_vl_utils 0.0.14+ reqired
    image_inputs, video_inputs, video_kwargs = process_vision_info(
        messages,
        image_patch_size=processor.image_processor.patch_size,
        return_video_kwargs=True,
        return_video_metadata=True
    )
    print(f"video_kwargs: {video_kwargs}")

    mm_data = {}
    if image_inputs is not None:
        mm_data['image'] = image_inputs
    if video_inputs is not None:
        mm_data['video'] = video_inputs

    return {
        'prompt': text,
        'multi_modal_data': mm_data,
        'mm_processor_kwargs': video_kwargs
    }


if __name__ == '__main__':
    # messages = [
    #     {
    #         "role": "user",
    #         "content": [
    #             {
    #                 "type": "video",
    #                 "video": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen2-VL/space_woaudio.mp4",
    #             },
    #             {"type": "text", "text": "这段视频有多长"},
    #         ],
    #     }
    # ]

    messages = [
        {
            "role": "user",
            "content": [
              {
                  "type": "image",
                  "image": "https://ofasys-multimodal-wlcb-3-toshanghai.oss-accelerate.aliyuncs.com/wpf272043/keepme/image/receipt.png",
              },
              {"type": "text", "text": "Read all the text in the image."},
            ],
        }
    ]

    # TODO: change to your own checkpoint path
    checkpoint_path = "../weights/Qwen2.5-VL-3B-Instruct-AWQ"
    processor = AutoProcessor.from_pretrained(checkpoint_path)
    inputs = [prepare_inputs_for_vllm(message, processor) for message in [messages]]

    llm = LLM(
        model=checkpoint_path,
        max_model_len=2048,
        mm_encoder_tp_mode="data",
        seed=0
    )

    sampling_params = SamplingParams(
        temperature=0,
        max_tokens=1024,
        top_k=-1,
        stop_token_ids=[],
    )

    for i, input_ in enumerate(inputs):
        print()
        print('=' * 40)
        print(f"Inputs[{i}]: {input_['prompt']=!r}")
    print('\n' + '>' * 40)

    outputs = llm.generate(inputs, sampling_params=sampling_params)
    for i, output in enumerate(outputs):
        generated_text = output.outputs[0].text
        print()
        print('=' * 40)
        print(f"Generated text: {generated_text!r}")

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 10-22 11:13:59 [__init__.py:216] Automatically detected platform cuda.


The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


video_kwargs: {'do_sample_frames': False}
INFO 10-22 11:14:02 [utils.py:233] non-default args: {'seed': 0, 'max_model_len': 2048, 'disable_log_stats': True, 'mm_encoder_tp_mode': 'data', 'model': '../weights/Qwen2.5-VL-3B-Instruct-AWQ'}
INFO 10-22 11:14:02 [model.py:547] Resolved architecture: Qwen2_5_VLForConditionalGeneration


`torch_dtype` is deprecated! Use `dtype` instead!


...

(EngineCore_DP0 pid=177055) INFO 10-22 11:14:11 [default_loader.py:267] Loading weights took 0.66 seconds
(EngineCore_DP0 pid=177055) INFO 10-22 11:14:12 [gpu_model_runner.py:2653] Model loading took 3.3169 GiB and 0.914245 seconds
(EngineCore_DP0 pid=177055) INFO 10-22 11:14:12 [gpu_model_runner.py:3344] Encoder cache will be initialized with a budget of 16384 tokens, and profiled with 1 image items of the maximum feature size.
(EngineCore_DP0 pid=177055) INFO 10-22 11:14:20 [backends.py:548] Using cache directory: /root/.cache/vllm/torch_compile_cache/3116fa2338/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=177055) INFO 10-22 11:14:20 [backends.py:559] Dynamo bytecode transform time: 4.65 s
(EngineCore_DP0 pid=177055) INFO 10-22 11:14:22 [backends.py:197] Cache the graph for dynamic shape for later use
(EngineCore_DP0 pid=177055) INFO 10-22 11:14:36 [backends.py:218] Compiling a graph for dynamic shape takes 15.68 s
(EngineCore_DP0 pid=177055) INFO 10-22 11:14:45 [mo

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:02<00:00, 27.30it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 34.09it/s]


(EngineCore_DP0 pid=177055) INFO 10-22 11:14:50 [gpu_model_runner.py:3480] Graph capturing finished in 4 secs, took 0.74 GiB
(EngineCore_DP0 pid=177055) INFO 10-22 11:14:50 [core.py:210] init engine (profile, create kv cache, warmup model) took 38.29 seconds
INFO 10-22 11:14:50 [llm.py:306] Supported_tasks: ['generate']

Inputs[0]: input_['prompt']='<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\n<|vision_start|><|image_pad|><|vision_end|>Read all the text in the image.<|im_end|>\n<|im_start|>assistant\n'

>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>


Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.86it/s, est. speed input: 602.02 toks/s, output: 137.92 toks/s]


Generated text: "Auntie Anne's\nCINNAMON SUGAR\n1 x 17.000\n17,000\nSUB TOTAL\n17,000\nGRAND TOTAL\n17,000\nCASH IDR\n20,000\nCHANGE DUE\n3,000"


In [ ]:
CUDA_LAUNCH_BLOCKING=1 vllm serve ../weights/Qwen2.5-VL-3B-Instruct-AWQ \
  --async-scheduling \
  --max-model-len 6144 \
  --host 0.0.0.0 \
  --port 22002 \
  --gpu-memory-utilization 0.90 \
  --allowed-local-media-path /root/backup/workspace/MS-AI-1000/v1.6.0-dev \
  --trust-remote-code

  --tokenizer-mode auto


In [7]:
import time
from openai import OpenAI
from pathlib import Path

client = OpenAI(
    api_key="EMPTY",
    base_url="http://127.0.0.1:22002/v1",
    timeout=3600
)

# 1. Define the path to your video file.
#    Replace with the actual path on your system.
video_file_path_str = "./test.avi" 

# 2. Get the absolute path and format it as a file URI.
video_path = Path(video_file_path_str).resolve()
file_uri = video_path.as_uri()

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "video_url",
                "video_url": {
                    "url": file_uri
                }
            },
            {
                "type": "text",
                "text": "How long is this video?"
            }
        ]
    }
]

start = time.time()
response = client.chat.completions.create(
    model="../weights/Qwen2.5-VL-3B-Instruct-AWQ",
    messages=messages,
    max_tokens=2048
)

print(f"Response costs: {time.time() - start:.2f}s")
print(f"Generated text: {response.choices[0].message.content}")

APIConnectionError: Connection error.

In [9]:
import time
from openai import OpenAI
from pathlib import Path

client = OpenAI(
    api_key="EMPTY",
    base_url="http://127.0.0.1:22002/v1",
    timeout=3600
)

# 1. Define the path to your video file.
#    Replace with the actual path on your system.
video_file_path_str = "../test.mp4" 

# 2. Get the absolute path and format it as a file URI.
video_path = Path(video_file_path_str).resolve()
file_uri = video_path.as_uri()

# messages = [
#     {
#         "role": "user",
#         "content": [
#             {
#                 "type": "video_url",
#                 "video_url": {
#                     "url": file_uri
#                 }
#             },
#             {
#                 "type": "text",
#                 "text": "How long is this video?"
#             }
#         ]
#     }
# ]

messages = [
    {
        "role": "user",
        "content": [

            {
                "type": "text",
                "text": "hello"
            }
        ]
    }
]

start = time.time()
response = client.chat.completions.create(
    model="../weights/Qwen2.5-VL-3B-Instruct-AWQ",
    messages=messages,
    max_tokens=40

)

print(f"Response costs: {time.time() - start:.2f}s")
print(f"Generated text: {response.choices[0].message.content}")

InternalServerError: Error code: 500 - {'error': {'message': 'EngineCore encountered an issue. See stack trace (above) for the root cause.', 'type': 'Internal Server Error', 'param': None, 'code': 500}}

In [26]:
import time
from openai import OpenAI
from pathlib import Path

client = OpenAI(
    api_key="EMPTY",
    base_url="http://127.0.0.1:22002/v1",
    timeout=3600
)

# 1. Define the path to your video file.
#    Replace with the actual path on your system.
video_file_path_str = "./videos/swim_3_65_2.mp4" 

# 2. Get the absolute path and format it as a file URI.
video_path = Path(video_file_path_str).resolve()
file_uri = video_path.as_uri()

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "video_url",
                "video_url": {
                    "url": file_uri
                }
            },
            {
                "type": "text",
                "text": """You are a specialized Vision-Language Model tasked with analyzing swimming pool CCTV footage to detect individuals at risk of drowning. You must meticulously identify signs of drowning by observing people's movements, postures, and interactions with the water. Always prioritize human safety and pay close attention to even subtle cues. You should watch the provided video clip and comprehensively analyze changes in a person's movement and state over time, rather than judging based on isolated moments. Your assessment should confirm if continuous behavioral patterns align with drowning indicators.
Please determine if the person in the provided video is at risk of drowning. If **even one** of the following four drowning patterns is detected, output 'Drowning risk: Yes'. Otherwise, output 'Drowning risk: No'. Provide a description and reason for your judgment in JSON format.

**Drowning Indication Patterns:**

1.  **Surface Struggling**
    * Both arms are thrust upward, frantically trying to lift the body out of the water.
    * Noticeable bubbles or splashing around the hands and arms.

2.  **Stationary Flailing Swim**
    * The torso moves as if swimming, but the person remains in nearly the same horizontal position.
    * Hands repeatedly rise to head level and then slap the water downward in an attempt to push upward.

3.  **Help-Seeking Gestures**
    * One or both hands wave side-to-side above the head as a signal for help.
    * The mouth barely clears the surface and may form shapes resembling “Help”.

4.  **Rapid Submerge-Emerge Cycle**
    * The head sinks below the surface and then quickly rises again using arm movements.
    * This submerge-emerge pattern repeats urgently at short intervals (about 1-3 seconds).
    * When the face is submerged, movements like opening the mouth or pinching the nose as if trying to breathe may be observed.

5.  **Behavior that attempts to rise above the water**
    * This pattern may overlap with patterns 1-4, but essentially includes any inefficient or desperate movements aimed at buoyancy.

**Output JSON format (exactly one line):**

{description: <detailed description of observed behavior>, reason: \"<drowning pattern number or name detected (e.g., 'Rapid Submerge-Emerge Cycle') and explanation of specific visual cues matching that pattern>\", Drowning risk: Yes}
or
{description: <detailed description of observed behavior>, reason: \"No drowning indication pattern detected. Judged as normal swimming or water play behavior.\", Drowning risk: No}
"""
            }
        ]
    }
]

# messages = [
#     {
#         "role": "user",
#         "content": [

#             {
#                 "type": "text",
#                 "text": "hello"
#             }
#         ]
#     }
# ]

start = time.time()
response = client.chat.completions.create(
    model="Qwen/Qwen3-VL-4B-Instruct-FP8",
    messages=messages,
    max_tokens=1024

)

print(f"Response costs: {time.time() - start:.2f}s")
print(f"Generated text: {response.choices[0].message.content}")


Response costs: 1.81s
Generated text: {description: "The individuals in the video are actively swimming and moving in the pool, with some holding inflatable tubes and others engaging in typical swimming motions. There are no signs of struggling, flailing, or desperate attempts to rise above the water.", reason: "No drowning indication pattern detected. Judged as normal swimming or water play behavior.", Drowning risk: No}


In [ ]:
# Efficient inference with FP8 checkpoint
# Requires NVIDIA H100+ and CUDA 12+
vllm serve Qwen/Qwen3-VL-4B-Instruct-FP8 \
  --tensor-parallel-size 1 \
  --mm-encoder-tp-mode data \
  --enable-expert-parallel \
  --async-scheduling \
  --host 0.0.0.0 \
  --port 22002



vllm serve Qwen/Qwen3-VL-4B-Instruct-FP8 \
  --mm-encoder-tp-mode data \
  --async-scheduling \
  --host 0.0.0.0 \
  --port 22002 \
  --max-model-len 5120 \
  --gpu-memory-utilization 0.90 \
  --allowed-local-media-path /root/backup/workspace/MS-AI-1000/v1.6.0-dev \

In [1]:
import requests
from requests.auth import HTTPBasicAuth

auth = HTTPBasicAuth("admin", "1234") # NVR에 대한 ID / PW
event_info_post = f'http://192.168.0.102/api/events?types=70&since=2025-10-18t00:00:00&until=2025-10-22t00:00:00&devices=0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16&sort=0&total=true'

r = requests.get(event_info_post, auth=auth, timeout= 1)

event_info_ori = r.json()


# save_period = setting_info["VIDEO_SAVE"]["period"]
# today = datetime.today()


In [2]:
event_info_ori

{'total': 10,
 'offset': 0,
 'limit': 10,
 'events': [{'type': 70,
   'timestamp': 1761031681,
   'rowid': 150845,
   'devices': [13],
   'micro_ai': {'type': 6, 'object': 1, 'direction': 0}},
  {'type': 70,
   'timestamp': 1761031676,
   'rowid': 150844,
   'devices': [13],
   'micro_ai': {'type': 6, 'object': 1, 'direction': 0}},
  {'type': 70,
   'timestamp': 1761031345,
   'rowid': 150843,
   'devices': [13],
   'micro_ai': {'type': 6, 'object': 1, 'direction': 0}},
  {'type': 70,
   'timestamp': 1761031344,
   'rowid': 150842,
   'devices': [13],
   'micro_ai': {'type': 6, 'object': 1, 'direction': 0}},
  {'type': 70,
   'timestamp': 1761029826,
   'rowid': 150839,
   'devices': [3],
   'micro_ai': {'type': 6, 'object': 1, 'direction': 0}},
  {'type': 70,
   'timestamp': 1761029710,
   'rowid': 150838,
   'devices': [3],
   'micro_ai': {'type': 6, 'object': 1, 'direction': 0}},
  {'type': 70,
   'timestamp': 1760778180,
   'rowid': 147865,
   'devices': [5],
   'micro_ai': {'type'

In [1]:
import requests
from requests.auth import HTTPBasicAuth

auth = HTTPBasicAuth("admin", "1234") # NVR에 대한 ID / PW
event_info_post = f'http://192.168.0.102/download/video1?start=2025-10-23T13:32:54&end=2025-10-23T13:33:04'

r = requests.get(event_info_post, auth=auth, timeout= 1)


In [5]:
import requests
from requests.auth import HTTPBasicAuth
import sys

# --- 설정 ---
NVR_IP = "192.168.0.102"
USERNAME = "admin"
PASSWORD = "1234"

CAMERA_NUM = 1
START_TIME = "2025-10-23T13:32:54"
END_TIME = "2025-10-23T13:33:04"

OUTPUT_FILENAME = "downloaded_video.mp4"  # 저장할 파일 이름

# --- URL 구성 ---
# Docx에 따라 'index=1' (일반 MP4) 파라미터를 추가
url = f"http://{NVR_IP}/download/video{CAMERA_NUM}.mp4?start={START_TIME}&end={END_TIME}&index=1"

auth = HTTPBasicAuth(USERNAME, PASSWORD)

print(f"다운로드 시도: {url}")

try:
    # 1. stream=True: 응답 본문을 즉시 다운로드하지 않고 스트림으로 연결
    # 2. timeout=10: (연결) 타임아웃을 10초로 설정 (1초는 너무 짧음)
    with requests.get(url, auth=auth, stream=True, timeout=10) as r:
        
        # HTTP 오류 (401 인증 실패, 404 찾을 수 없음 등)가 발생하면 예외를 일으킴
        r.raise_for_status() 
        
        print(f"다운로드 시작... -> {OUTPUT_FILENAME}")
        
        # 'wb' (write binary) 모드로 파일을 엽니다.
        with open(OUTPUT_FILENAME, 'wb') as f:
            # 8192 바이트 (8KB) 씩 조각내어 다운로드
            for chunk in r.iter_content(chunk_size=8192): 
                if chunk: # chunk가 비어있지 않으면 파일에 씀
                    f.write(chunk)
                    
        print(f"다운로드 완료: {OUTPUT_FILENAME}")

except requests.exceptions.HTTPError as err:
    print(f"HTTP 오류 발생: {err}")
except requests.exceptions.ConnectionError as err:
    print(f"연결 오류 발생: {err} (IP 주소나 네트워크를 확인하세요)")
except requests.exceptions.Timeout as err:
    print(f"타임아웃 오류 발생: {err} (서버 응답이 너무 느립니다)")
except requests.exceptions.RequestException as err:
    print(f"알 수 없는 오류 발생: {err}")

다운로드 시도: http://192.168.0.102/download/video1.mp4?start=2025-10-23T13:32:54&end=2025-10-23T13:33:04&index=1
다운로드 시작... -> downloaded_video.mp4
다운로드 완료: downloaded_video.mp4
